# ==============================================================================
# REVISED: ATAC-seq + ChIP-seq Annotation
# Changes:
#   1. TSS window: 2kb upstream, 1kb downstream
#   2. All genes in window (not just nearest)
# ==============================================================================

In [5]:


library(GenomicRanges)
library(GenomicFeatures)
library(rtracklayer)
library(dplyr)
library(stringr)

base_dir = "/home/users/adhal/CorticalNeuronFate/CellConversionNSC"
# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

atac_dir <- file.path(base_dir, "data/atac_seq/ATAC-seq")
chip_path <- file.path(base_dir, "data/chip_seq/Oth.Neu.05.AllAg.AllCell.bed")
gtf_path <- file.path(base_dir, "data/annotations/gencode.vM36.annotation.gtf")
out_dir <- file.path(base_dir, "data/integrated")

# ------------------------------------------------------------------------------
# LOAD SAMPLE METADATA FROM FILE
# ------------------------------------------------------------------------------

sample_metadata_raw <- read.csv(file.path(base_dir, "data/atac_seq/ATAC-seq/samples_all.csv"))

# Rename columns to match what the script expects
sample_metadata <- data.frame(
    sample = sample_metadata_raw$SampleID,
    condition = sample_metadata_raw$Condition,
    tissue = sample_metadata_raw$Tissue,
    factor = sample_metadata_raw$Factor,
    replicate = sample_metadata_raw$Replicate,
    stringsAsFactors = FALSE
)

print(paste("Loaded", nrow(sample_metadata), "samples from CSV"))

# Filter to samples with existing peak files
sample_metadata$peak_file <- file.path(atac_dir, paste0(sample_metadata$sample, "_peaks.narrowPeak"))
sample_metadata$file_exists <- file.exists(sample_metadata$peak_file)

missing <- sample_metadata$sample[!sample_metadata$file_exists]
if (length(missing) > 0) {
    print(paste("Missing peak files for:", paste(missing, collapse = ", ")))
}

sample_metadata <- sample_metadata[sample_metadata$file_exists, ]
sample_metadata <- sample_metadata[, c("sample", "condition", "tissue", "factor", "replicate")]

print(paste("Samples with peak files:", nrow(sample_metadata)))
print(sample_metadata)
# ------------------------------------------------------------------------------
# STEP 1: Load promoters from GTF (2kb upstream, 1kb downstream)
# ------------------------------------------------------------------------------

print("Loading gene annotation...")
txdb <- makeTxDbFromGFF(gtf_path, format = "gtf")
genes_gr <- genes(txdb)

# CHANGE 1: TSS window = 2kb upstream, 1kb downstream
promoters_gr <- promoters(genes_gr, upstream = 2000, downstream = 1000)

gtf <- import(gtf_path)
gtf_genes <- gtf[gtf$type == "gene"]
gene_map <- data.frame(
    gene_id = gtf_genes$gene_id,
    gene_name = gtf_genes$gene_name,
    stringsAsFactors = FALSE
)
gene_map$gene_id_clean <- gsub("\\..*", "", gene_map$gene_id)

promoters_gr$gene_id_clean <- gsub("\\..*", "", names(promoters_gr))
promoters_gr$symbol <- gene_map$gene_name[match(promoters_gr$gene_id_clean, gene_map$gene_id_clean)]

print(paste("Promoters (2kb upstream, 1kb downstream):", length(promoters_gr)))

# ------------------------------------------------------------------------------
# STEP 2: Load ATAC-seq peaks
# ------------------------------------------------------------------------------

print("Loading ATAC-seq peaks...")

load_atac_peaks <- function(sample_name, atac_dir) {
    file_path <- file.path(atac_dir, paste0(sample_name, "_peaks.narrowPeak"))
    peaks <- read.table(file_path, sep = "\t", header = FALSE)
    colnames(peaks) <- c("chr", "start", "end", "name", "score", "strand",
                         "signalValue", "pValue", "qValue", "peak")
    gr <- GRanges(
        seqnames = peaks$chr,
        ranges = IRanges(start = peaks$start, end = peaks$end),
        strand = "*",
        score = peaks$score,
        signalValue = peaks$signalValue,
        sample = sample_name
    )
    return(gr)
}

atac_list <- lapply(sample_metadata$sample, function(s) {
    print(paste("  Loading", s))
    load_atac_peaks(s, atac_dir)
})
atac_all <- do.call(c, atac_list)
print(paste("Total ATAC peaks:", length(atac_all)))

# ------------------------------------------------------------------------------
# STEP 3: Load ChIP-seq peaks
# ------------------------------------------------------------------------------

print("Loading ChIP-seq peaks...")

chip_df <- read.table(chip_path, sep = "\t", header = FALSE, skip = 1, 
                       quote = "", fill = TRUE,
                       colClasses = c("character", "integer", "integer", "character", 
                                     "integer", "character", "integer", "integer", "character"))
colnames(chip_df) <- c("chr", "start", "end", "metadata", "score", "strand",
                        "thickStart", "thickEnd", "rgb")

chip_df$TF <- str_extract(chip_df$metadata, "(?<=Name=)[^%]+")
chip_df$TF <- trimws(chip_df$TF)
chip_df$cell_type <- str_extract(chip_df$metadata, "(?<=@%20)[^)]+")
chip_df$cell_type <- gsub("%20", " ", chip_df$cell_type)

chip_gr <- GRanges(
    seqnames = chip_df$chr,
    ranges = IRanges(start = chip_df$start, end = chip_df$end),
    strand = "*",
    TF = chip_df$TF,
    score = chip_df$score,
    cell_type = chip_df$cell_type
)

print(paste("Total ChIP peaks:", length(chip_gr)))
print(paste("Unique TFs:", length(unique(chip_df$TF))))

# ------------------------------------------------------------------------------
# STEP 4: Annotate ATAC peaks to ALL genes in TSS window
# ------------------------------------------------------------------------------

print("Annotating ATAC peaks to promoters (all genes in window)...")

# CHANGE 2: findOverlaps returns ALL overlapping genes
atac_hits <- findOverlaps(atac_all, promoters_gr, type = "any")

atac_annotated <- data.frame(
    chr = as.character(seqnames(atac_all))[queryHits(atac_hits)],
    start = start(atac_all)[queryHits(atac_hits)],
    end = end(atac_all)[queryHits(atac_hits)],
    score = mcols(atac_all)$score[queryHits(atac_hits)],
    signalValue = mcols(atac_all)$signalValue[queryHits(atac_hits)],
    sample = mcols(atac_all)$sample[queryHits(atac_hits)],
    gene = promoters_gr$symbol[subjectHits(atac_hits)],
    gene_id = promoters_gr$gene_id_clean[subjectHits(atac_hits)],
    stringsAsFactors = FALSE
)
atac_annotated$condition <- sample_metadata$condition[match(atac_annotated$sample, sample_metadata$sample)]
atac_annotated <- atac_annotated[!is.na(atac_annotated$gene), ]

print(paste("ATAC annotated rows:", nrow(atac_annotated)))
print(paste("Unique genes:", length(unique(atac_annotated$gene))))

# ------------------------------------------------------------------------------
# STEP 5: Annotate ChIP peaks to ALL genes in TSS window
# ------------------------------------------------------------------------------

print("Annotating ChIP peaks to promoters (all genes in window)...")

chip_hits <- findOverlaps(chip_gr, promoters_gr, type = "any")

chip_annotated <- data.frame(
    chr = as.character(seqnames(chip_gr))[queryHits(chip_hits)],
    start = start(chip_gr)[queryHits(chip_hits)],
    end = end(chip_gr)[queryHits(chip_hits)],
    score = mcols(chip_gr)$score[queryHits(chip_hits)],
    TF = mcols(chip_gr)$TF[queryHits(chip_hits)],
    cell_type = mcols(chip_gr)$cell_type[queryHits(chip_hits)],
    gene = promoters_gr$symbol[subjectHits(chip_hits)],
    gene_id = promoters_gr$gene_id_clean[subjectHits(chip_hits)],
    stringsAsFactors = FALSE
)
chip_annotated <- chip_annotated[!is.na(chip_annotated$gene), ]

print(paste("ChIP annotated rows:", nrow(chip_annotated)))
print(paste("Unique genes:", length(unique(chip_annotated$gene))))
print(paste("Unique TFs:", length(unique(chip_annotated$TF))))

# # ------------------------------------------------------------------------------
# # STEP 6: Save annotated files
# # ------------------------------------------------------------------------------

# print("Saving annotated files...")

# write.table(atac_annotated, file.path(out_dir, "atac_annotated.tsv"),
#             sep = "\t", row.names = FALSE, quote = FALSE)

# write.table(chip_annotated, file.path(out_dir, "chip_annotated.tsv"),
#             sep = "\t", row.names = FALSE, quote = FALSE)

# print("Done!")
# print(paste("ATAC annotated:", nrow(atac_annotated), "rows"))
# print(paste("ChIP annotated:", nrow(chip_annotated), "rows"))


# ==============================================================================
# STEP 7: Filter ChIP-seq to relevant cell types (Cortex + LGE/Ganglionic Eminence)
# ==============================================================================

print("Filtering ChIP-seq to cortex and LGE relevant cell types...")

# Check all cell types
print("All cell types in ChIP-seq:")
print(sort(unique(chip_annotated$cell_type)))

# Define relevant cell types for cortex and LGE
relevant_cell_types <- c(
    # Cortex-related
    "Cortex",
    "Brain cortex", 
    "Cerebral Cortex",
    "Cortical neuron",
    "Dorsal cortex",
    "Frontal cortex",
    "Prefrontal Cortex",
    "Visual Cortex",
    "Cortical midline",
    "Cortical oligodendrocyte progenitor",
    
    # LGE/Ganglionic eminence related
    "Ganglionic eminence",
    "Basal Ganglia",
    "Caudate putamen",
    "Striatal neurons",
    "Nucleus Accumbens",
    "Corpus Striatum",
    
    # General neural progenitors (relevant to both)
    "Neural Stem Cells",
    "Neural progenitor cells",
    "Neural precursors",
    "Neuroepithelial progenitor cell",
    "Neurospheres",
    "Forebrain",
    
    # General brain (if you want broader coverage)
    "Brain"
)

# Filter chip_annotated
chip_annotated_filtered <- chip_annotated[chip_annotated$cell_type %in% relevant_cell_types, ]

print(paste("ChIP annotated rows before filter:", nrow(chip_annotated)))
print(paste("ChIP annotated rows after filter:", nrow(chip_annotated_filtered)))
print(paste("Unique TFs after filter:", length(unique(chip_annotated_filtered$TF))))
print(paste("Unique genes after filter:", length(unique(chip_annotated_filtered$gene))))

print("Cell types kept:")
print(table(chip_annotated_filtered$cell_type))

# ==============================================================================
# STEP 8: Create ChIP-ATAC overlap (maxgap = 1bp)
# ==============================================================================

print("Creating ChIP-ATAC overlap (maxgap = 1bp)...")

# Convert to GRanges
chip_gr_filtered <- GRanges(
    seqnames = chip_annotated_filtered$chr,
    ranges = IRanges(start = chip_annotated_filtered$start, end = chip_annotated_filtered$end),
    strand = "*",
    score = chip_annotated_filtered$score,
    TF = chip_annotated_filtered$TF,
    cell_type = chip_annotated_filtered$cell_type,
    gene = chip_annotated_filtered$gene,
    gene_id = chip_annotated_filtered$gene_id
)

atac_gr <- GRanges(
    seqnames = atac_annotated$chr,
    ranges = IRanges(start = atac_annotated$start, end = atac_annotated$end),
    strand = "*",
    score = atac_annotated$score,
    signalValue = atac_annotated$signalValue,
    sample = atac_annotated$sample,
    gene = atac_annotated$gene,
    gene_id = atac_annotated$gene_id,
    condition = atac_annotated$condition
)

# Find overlaps with maxgap = 1bp
overlap_hits <- findOverlaps(chip_gr_filtered, atac_gr, maxgap = 1, type = "any")

print(paste("Total overlaps (maxgap=1):", length(overlap_hits)))

# Get genes from both sides
chip_genes <- mcols(chip_gr_filtered)$gene[queryHits(overlap_hits)]
atac_genes <- mcols(atac_gr)$gene[subjectHits(overlap_hits)]

# Filter to same gene annotation
same_gene <- chip_genes == atac_genes
overlap_hits_filtered <- overlap_hits[same_gene]

print(paste("Overlaps with same gene:", sum(same_gene)))

# ==============================================================================
# STEP 9: Build overlap dataframe
# ==============================================================================

print("Building overlap dataframe...")

overlap_df <- data.frame(
    # ChIP info
    chip_chr = as.character(seqnames(chip_gr_filtered))[queryHits(overlap_hits_filtered)],
    chip_start = start(chip_gr_filtered)[queryHits(overlap_hits_filtered)],
    chip_end = end(chip_gr_filtered)[queryHits(overlap_hits_filtered)],
    chip_score = mcols(chip_gr_filtered)$score[queryHits(overlap_hits_filtered)],
    TF = mcols(chip_gr_filtered)$TF[queryHits(overlap_hits_filtered)],
    cell_type = mcols(chip_gr_filtered)$cell_type[queryHits(overlap_hits_filtered)],
    
    # ATAC info
    atac_chr = as.character(seqnames(atac_gr))[subjectHits(overlap_hits_filtered)],
    atac_start = start(atac_gr)[subjectHits(overlap_hits_filtered)],
    atac_end = end(atac_gr)[subjectHits(overlap_hits_filtered)],
    atac_score = mcols(atac_gr)$score[subjectHits(overlap_hits_filtered)],
    atac_signal = mcols(atac_gr)$signalValue[subjectHits(overlap_hits_filtered)],
    sample = mcols(atac_gr)$sample[subjectHits(overlap_hits_filtered)],
    condition = mcols(atac_gr)$condition[subjectHits(overlap_hits_filtered)],
    
    # Gene info
    gene = mcols(chip_gr_filtered)$gene[queryHits(overlap_hits_filtered)],
    gene_id = mcols(chip_gr_filtered)$gene_id[queryHits(overlap_hits_filtered)],
    
    stringsAsFactors = FALSE
)

# Remove NA genes
overlap_df <- overlap_df[!is.na(overlap_df$gene), ]

print(paste("Final overlap rows:", nrow(overlap_df)))
print(paste("Unique TF-gene pairs:", nrow(unique(overlap_df[, c("TF", "gene")]))))
print(paste("Unique TFs:", length(unique(overlap_df$TF))))
print(paste("Unique genes:", length(unique(overlap_df$gene))))

# ==============================================================================
# STEP 10: Summary statistics
# ==============================================================================

print("\n=== Overlap Summary ===")

# TFs per gene
tf_per_gene <- overlap_df %>%
    group_by(gene) %>%
    summarise(n_tfs = n_distinct(TF), .groups = "drop")
print(paste("Mean TFs per gene:", round(mean(tf_per_gene$n_tfs), 2)))

# Genes per TF
genes_per_tf <- overlap_df %>%
    group_by(TF) %>%
    summarise(n_genes = n_distinct(gene), .groups = "drop")
print(paste("Mean genes per TF:", round(mean(genes_per_tf$n_genes), 2)))

# By condition
print("\nOverlaps by condition:")
print(table(overlap_df$condition))

# Top TFs by number of target genes
print("\nTop 20 TFs by target genes:")
print(head(genes_per_tf %>% arrange(desc(n_genes)), 20))

# # ==============================================================================
# # STEP 11: Save files
# # ==============================================================================

# print("Saving files...")

# # Save filtered chip_annotated
# write.table(chip_annotated_filtered, file.path(out_dir, "chip_annotated_filtered.tsv"),
#             sep = "\t", row.names = FALSE, quote = FALSE)

# # Save overlap_df
# write.table(overlap_df, file.path(out_dir, "overlap_annotated.tsv"),
#             sep = "\t", row.names = FALSE, quote = FALSE)

# print("Done!")
# print(paste("ChIP annotated filtered:", nrow(chip_annotated_filtered), "rows"))
# print(paste("Overlap df:", nrow(overlap_df), "rows"))

[1] "Loaded 22 samples from CSV"
[1] "Missing peak files for: ISF1045, ISF1046, ISF1056"
[1] "Samples with peak files: 19"
    sample condition tissue   factor replicate
3  ISF1047       E14 Cortex PSA-NCAM         2
4  ISF1048       E14    LGE    CD133         1
5  ISF1049       E18 Cortex    CD133         1
6  ISF1050       E18    LGE    CD133         1
7  ISF1051       E14 Cortex PSA-NCAM         3
8  ISF1052       E14    LGE    CD133         2
9  ISF1053       E14 Cortex    CD133         2
10 ISF1054       E14    LGE PSA-NCAM         1
11 ISF1055       E18 Cortex    CD133         2
13 ISF1057       E14 Cortex    CD133         3
14 ISF1058       E14    LGE PSA-NCAM         2
15 ISF1059       E14    LGE    CD133         3
16 ISF1060       E18    LGE    CD133         2
17 ISF1061       E18 Cortex    CD133         4
18 ISF1062       E14    LGE PSA-NCAM         3
19 ISF1063       E14    LGE    CD133         4
20 ISF1064       E14    LGE PSA-NCAM         4
21 ISF1065       E18    LGE    

Warning message in call_fun_in_txdbmaker("makeTxDbFromGFF", ...):
“makeTxDbFromGFF() has moved to the txdbmaker package. Please call
  txdbmaker::makeTxDbFromGFF() to get rid of this warning.”
Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
“The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored.”
OK



[1] "Promoters (2kb upstream, 1kb downstream): 78239"
[1] "Loading ATAC-seq peaks..."
[1] "  Loading ISF1047"
[1] "  Loading ISF1048"
[1] "  Loading ISF1049"
[1] "  Loading ISF1050"
[1] "  Loading ISF1051"
[1] "  Loading ISF1052"
[1] "  Loading ISF1053"
[1] "  Loading ISF1054"
[1] "  Loading ISF1055"
[1] "  Loading ISF1057"
[1] "  Loading ISF1058"
[1] "  Loading ISF1059"
[1] "  Loading ISF1060"
[1] "  Loading ISF1061"
[1] "  Loading ISF1062"
[1] "  Loading ISF1063"
[1] "  Loading ISF1064"
[1] "  Loading ISF1065"
[1] "  Loading ISF1066"


Warning message in .merge_two_Seqinfo_objects(x, y):
“Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': chrUn_GL456372v1, chrUn_GL456394v1
  - in 'y': chr5_JH584299v1_random
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning).”
Warning message in .merge_two_Seqinfo_objects(x, y):
“Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': chr5_JH584299v1_random
  - in 'y': chr5_GL456354v1_random, chr7_GL456219v1_random
  Make sure to always combine/compare objects based on the same reference
  genome (use suppressWarnings() to suppress this warning).”
Warning message in .merge_two_Seqinfo_objects(x, y):
“Each of the 2 combined objects has sequence levels not in the other:
  - in 'x': chr5_JH584299v1_random, chr5_GL456354v1_random, chr7_GL456219v1_random
  - in 'y': chrUn_GL456367v1
  Make sure to always combine/compare objects based on the same reference

[1] "Total ATAC peaks: 786550"
[1] "Loading ChIP-seq peaks..."
[1] "Total ChIP peaks: 10145526"
[1] "Unique TFs: 197"
[1] "Annotating ATAC peaks to promoters (all genes in window)..."
[1] "ATAC annotated rows: 483985"
[1] "Unique genes: 32791"
[1] "Annotating ChIP peaks to promoters (all genes in window)..."
[1] "ChIP annotated rows: 1497928"
[1] "Unique genes: 60943"
[1] "Unique TFs: 197"
[1] "Filtering ChIP-seq to cortex and LGE relevant cell types..."
[1] "All cell types in ChIP-seq:"
 [1] "Arcuate Nucleus of Hypothalamus"     
 [2] "Astrocytes"                          
 [3] "AtT-20"                              
 [4] "Basal Ganglia"                       
 [5] "Brain"                               
 [6] "Brain cortex"                        
 [7] "Brain tumor stem cells"              
 [8] "Brain tumors"                        
 [9] "BV-2"                                
[10] "C17-2"                               
[11] "CAD"                                 
[12] "Caudate putamen" 

In [7]:
write.table(atac_annotated, file.path(out_dir, "atac_annotated.tsv"),
            sep = "\t", row.names = FALSE, quote = FALSE)

write.table(chip_annotated, file.path(out_dir, "chip_annotated.tsv"),
            sep = "\t", row.names = FALSE, quote = FALSE)

# Save filtered chip_annotated
write.table(chip_annotated_filtered, file.path(out_dir, "chip_annotated_filtered.tsv"),
            sep = "\t", row.names = FALSE, quote = FALSE)

write.table(overlap_df, file.path(out_dir, "overlap_annotated.tsv"),
            sep = "\t", row.names = FALSE, quote = FALSE)